In [ ]:
import os
DATA_ROOT = os.environ.get('WRH_DATA_ROOT', 'G:/WRH_data')  # local data folder; see README
import os, re, glob
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.lines import Line2D
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import rasterio
from rasterio.warp import transform_bounds

# ----------------------- Config -----------------------
MODE      = "raw"     # "raw"=inspect raw state (extracted+east_asia_CMTDS); "organized"=inspect the ORGANIZED folder
BASE      = rf"{DATA_ROOT}/climate_AIS_data/GMTDS"
RAW_DIRS  = [os.path.join(BASE, "extracted"), os.path.join(BASE, "east_asia_CMTDS")]
ORGANIZED = os.path.join(BASE, "ORGANIZED")
YEARS     = range(2011, 2022)     # 2011-2021, full coverage = 132 months
CLUSTER_D = 1.5                   # centers closer than this (deg) count as the same sea
# ----------------------------------------------------

MON_RE = re.compile(r"_(\d{4})_(\d{2})\.tif$", re.I)

def latlon_bounds(ds):
    b = ds.bounds
    try:
        return transform_bounds(ds.crs, "EPSG:4326", b.left, b.bottom, b.right, b.top)
    except Exception:
        return (b.left, b.bottom, b.right, b.top)

def scan_tifs(dirs):
    """Walk all tifs under the directory tree -> [(west,south,east,north, year, month), ...]"""
    recs = []
    for d in dirs:
        for t in glob.glob(os.path.join(d, "**", "*.tif"), recursive=True):
            m = MON_RE.search(os.path.basename(t))
            if not m:
                continue
            y, mo = int(m.group(1)), int(m.group(2))
            if y not in YEARS:
                continue
            with rasterio.open(t) as ds:
                w, s, e, n = latlon_bounds(ds)
            recs.append((round(w, 3), round(s, 3), round(e, 3), round(n, 3), y, mo))
    return recs

def cluster(recs):
    """Cluster into seas by bbox center -> [(representative bbox, unique months, distinct extents in this sea), ...]"""
    clusters = []
    for w, s, e, n, y, mo in recs:
        cx, cy = (w + e) / 2, (s + n) / 2
        hit = None
        for c in clusters:
            if abs(c["cx"] - cx) <= CLUSTER_D and abs(c["cy"] - cy) <= CLUSTER_D:
                hit = c; break
        if hit is None:
            hit = {"cx": cx, "cy": cy, "bboxes": {}, "months": set()}
            clusters.append(hit)
        hit["bboxes"][(w, s, e, n)] = hit["bboxes"].get((w, s, e, n), 0) + 1
        hit["months"].add((y, mo))
    out = []
    for c in clusters:
        rep = max(c["bboxes"].items(), key=lambda kv: kv[1])[0]   # most frequent bbox
        out.append((rep, len(c["months"]), len(c["bboxes"])))
    return out

# ---- scan + cluster ----
dirs = RAW_DIRS if MODE == "raw" else [ORGANIZED]
print(f"MODE={MODE}  scanning: {dirs}")
recs = scan_tifs(dirs)
print(f"read {len(recs)} tifs ({min(YEARS)}-{max(YEARS)})")
regions = cluster(recs)
print(f"merged into {len(regions)} actual seas")

# ---- draw world map ----
fig = plt.figure(figsize=(20, 10.5))
ax = plt.axes(projection=ccrs.PlateCarree())
ax.set_global()
ax.add_feature(cfeature.LAND, facecolor="#f0ece1")
ax.add_feature(cfeature.OCEAN, facecolor="#dceaf2")
ax.add_feature(cfeature.COASTLINE, linewidth=0.4, edgecolor="#999")
ax.add_feature(cfeature.BORDERS, linewidth=0.2, edgecolor="#c8c8c8")
ax.gridlines(draw_labels=True, linewidth=0.3, color="gray", alpha=0.35)

full = 132  # 2011-2021 full month count
regions.sort(key=lambda r: -((r[0][2] - r[0][0]) * (r[0][3] - r[0][1])))  # draw large boxes first
for rep, nmonths, nbbox in regions:
    w, s, e, n = rep
    if nmonths >= full:
        color, ls, lw = "#2e7d32", "-", 1.6            # green: complete in time
    elif nmonths >= full * 0.5:
        color, ls, lw = "#ef8c00", (0, (5, 2)), 2.4    # orange: partial / extent drift
    else:
        color, ls, lw = "#d32f2f", "--", 2.4           # red: severely incomplete
    ax.add_patch(mpatches.Rectangle((w, s), e - w, n - s, transform=ccrs.PlateCarree(),
                 facecolor=color, edgecolor=color, alpha=0.20, linewidth=lw, linestyle=ls, zorder=5))
    ax.add_patch(mpatches.Rectangle((w, s), e - w, n - s, transform=ccrs.PlateCarree(),
                 facecolor="none", edgecolor=color, linewidth=lw, linestyle=ls, zorder=6))
    tag = f"{nmonths}/{full}" + ("" if nbbox == 1 else f"\n({nbbox} extents)")
    ax.text((w + e) / 2, (s + n) / 2, tag, transform=ccrs.PlateCarree(), fontsize=7.5,
            ha="center", va="center", weight="bold", color="black", zorder=7,
            bbox=dict(boxstyle="round,pad=0.12", fc="white", ec="none", alpha=0.7))

legend = [
    Line2D([0], [0], color="#2e7d32", lw=3, label="Complete (132/132 months)"),
    Line2D([0], [0], color="#ef8c00", lw=3, ls=(0, (5, 2)), label="Partial / extent shifted mid-series"),
    Line2D([0], [0], color="#d32f2f", lw=3, ls="--", label="Severely incomplete (<50%)"),
]
ax.legend(handles=legend, loc="lower left", fontsize=10, framealpha=0.9)
ax.set_title(f"GMTDS downloaded coverage ({MODE}) — {len(regions)} sea regions by REAL TIFF georeferencing\n"
             f"labels = unique months present in {min(YEARS)}-{max(YEARS)}", fontsize=13, weight="bold")
plt.tight_layout()
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.lines import Line2D
import cartopy.crs as ccrs
import cartopy.feature as cfeature

# Format: (south_lat, north_lat, west_lon, east_lon)
TARGET_BOXES = {
    # Extratropical storm tracks (ONDJFM): same regime, replicated in two oceans
    "NW_Europe":   [(43, 63, -15, 0), (48, 63, 0, 10)],   # Biscay+Channel+North Sea; east edge 10E ceded to Baltic
    "NW_Atlantic": [(18, 50, -98, -60)],                  # Gulf of Mexico+US East+Canadian Maritimes
    # Typhoon/monsoon: same hazard, different development levels (main adaptation contrast)
    "SE_Asia":     [(-10, 20, 95, 127)],                  # Malacca-S. China Sea-Indonesia; bounded at 20N
    "E_Asia":      [(20, 42, 117, 145)],                  # S. China-E. China Sea-Yellow/Bohai-Japan/Korea; north of 20N
    # Sheltered seas: endogenous low-sea-state control
    "Med_Black":   [(30, 43, -5.5, 0), (30, 47.5, 0, 42)],# east of Gibraltar, incl. Black Sea
    "Baltic":      [(53, 66, 10, 30)],                    # east of 10E, incl. Danish straits transition
}

COLORS = {
    "NW_Europe": "#1f77b4", "NW_Atlantic": "#2ca02c", "SE_Asia": "#d62728",
    "E_Asia": "#ff7f0e",    "Med_Black": "#9467bd",   "Baltic": "#17becf",
}

fig = plt.figure(figsize=(20, 10))
ax = plt.axes(projection=ccrs.PlateCarree())
ax.set_extent([-105, 150, -18, 72], crs=ccrs.PlateCarree())  # zoom to target regions; use ax.set_global() for global view
ax.add_feature(cfeature.LAND, facecolor="#f0ece1")
ax.add_feature(cfeature.OCEAN, facecolor="#dceaf2")
ax.add_feature(cfeature.COASTLINE, linewidth=0.5, edgecolor="#888")
ax.add_feature(cfeature.BORDERS, linewidth=0.25, edgecolor="#c8c8c8")
ax.gridlines(draw_labels=True, linewidth=0.3, color="gray", alpha=0.35)

for name, boxes in TARGET_BOXES.items():
    color = COLORS[name]
    for i, (s, n, w, e) in enumerate(boxes):
        ax.add_patch(mpatches.Rectangle((w, s), e - w, n - s, transform=ccrs.PlateCarree(),
                     facecolor=color, edgecolor=color, alpha=0.25, linewidth=2.2, zorder=5))
        ax.add_patch(mpatches.Rectangle((w, s), e - w, n - s, transform=ccrs.PlateCarree(),
                     facecolor="none", edgecolor=color, linewidth=2.2, zorder=6))
    # place label at the overall center of all sub-boxes of the region
    s0 = min(b[0] for b in boxes); n0 = max(b[1] for b in boxes)
    w0 = min(b[2] for b in boxes); e0 = max(b[3] for b in boxes)
    ax.text((w0 + e0) / 2, (s0 + n0) / 2, name, transform=ccrs.PlateCarree(),
            fontsize=11, ha="center", va="center", weight="bold", color="black", zorder=7,
            bbox=dict(boxstyle="round,pad=0.2", fc="white", ec=color, lw=1.5, alpha=0.85))

handles = [Line2D([0], [0], color=c, lw=3, label=k) for k, c in COLORS.items()]
ax.legend(handles=handles, loc="lower left", fontsize=11, framealpha=0.9, title="Target regions")
ax.set_title("GMTDS target study regions TARGET_BOXES  (format: south, north, west, east)",
             fontsize=14, weight="bold")
plt.tight_layout()
plt.show()


In [ ]:
import os, re, glob
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.lines import Line2D
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import rasterio
from rasterio.warp import transform_bounds

# ----------------------- Config -----------------------
MODE      = "raw"     # "raw"=inspect raw state (extracted+east_asia_CMTDS); "organized"=inspect the ORGANIZED folder
BASE      = rf"{DATA_ROOT}/climate_AIS_data/GMTDS"
RAW_DIRS  = [os.path.join(BASE, "extracted"), os.path.join(BASE, "east_asia_CMTDS")]
ORGANIZED = os.path.join(BASE, "ORGANIZED")
YEARS     = range(2011, 2022)     # 2011-2021, full coverage = 132 months
MERGE_FRAC = 0.5                  # bbox intersection >= this fraction of the smaller box => same sea
# ----------------------------------------------------

MON_RE = re.compile(r"_(\d{4})_(\d{2})\.tif$", re.I)

def latlon_bounds(ds):
    b = ds.bounds
    try:
        return transform_bounds(ds.crs, "EPSG:4326", b.left, b.bottom, b.right, b.top)
    except Exception:
        return (b.left, b.bottom, b.right, b.top)

def scan_tifs(dirs):
    """Walk all tifs under the directory tree -> [(west,south,east,north, year, month), ...]"""
    recs = []
    for d in dirs:
        for t in glob.glob(os.path.join(d, "**", "*.tif"), recursive=True):
            m = MON_RE.search(os.path.basename(t))
            if not m:
                continue
            y, mo = int(m.group(1)), int(m.group(2))
            if y not in YEARS:
                continue
            with rasterio.open(t) as ds:
                w, s, e, n = latlon_bounds(ds)
            recs.append((round(w, 3), round(s, 3), round(e, 3), round(n, 3), y, mo))
    return recs

# --- Key fix: merge by [bbox overlap fraction], not center distance ---
def _area(b):  return max(0, b[2]-b[0]) * max(0, b[3]-b[1])
def _inter(a, b):
    ix = (max(a[0],b[0]), max(a[1],b[1]), min(a[2],b[2]), min(a[3],b[3]))
    return _area(ix)
def _overlap_frac(a, b):
    sm = min(_area(a), _area(b))
    return _inter(a, b) / sm if sm > 0 else 0.0

def cluster(recs):
    """Merge into seas by bbox overlap -> [(union_bbox, unique months, distinct extents in this sea), ...]
    A sea whose extent drifts (high overlap) is merged back; adjacent distinct seas (small overlap) stay separate."""
    clusters = []
    for w, s, e, n, y, mo in recs:
        bb = (w, s, e, n)
        hit = None
        for c in clusters:
            if _overlap_frac(bb, c["union"]) >= MERGE_FRAC or _overlap_frac(bb, c["rep"]) >= MERGE_FRAC:
                hit = c; break
        if hit is None:
            hit = {"union": bb, "rep": bb, "bboxes": {}, "months": set()}
            clusters.append(hit)
        hit["bboxes"][bb] = hit["bboxes"].get(bb, 0) + 1
        hit["months"].add((y, mo))
        hit["union"] = (min(hit["union"][0], w), min(hit["union"][1], s),
                        max(hit["union"][2], e), max(hit["union"][3], n))
        hit["rep"] = max(hit["bboxes"].items(), key=lambda kv: kv[1])[0]
    return [(c["union"], len(c["months"]), len(c["bboxes"])) for c in clusters]

# ---- scan + cluster ----
dirs = RAW_DIRS if MODE == "raw" else [ORGANIZED]
print(f"MODE={MODE}  scanning: {dirs}")
recs = scan_tifs(dirs)
print(f"read {len(recs)} tifs ({min(YEARS)}-{max(YEARS)})")
regions = cluster(recs)
print(f"merged into {len(regions)} actual seas")

# ---- draw world map ----
fig = plt.figure(figsize=(20, 10.5))
ax = plt.axes(projection=ccrs.PlateCarree())
ax.set_global()
ax.add_feature(cfeature.LAND, facecolor="#f0ece1")
ax.add_feature(cfeature.OCEAN, facecolor="#dceaf2")
ax.add_feature(cfeature.COASTLINE, linewidth=0.4, edgecolor="#999")
ax.add_feature(cfeature.BORDERS, linewidth=0.2, edgecolor="#c8c8c8")
ax.gridlines(draw_labels=True, linewidth=0.3, color="gray", alpha=0.35)

full = 132  # 2011-2021 full month count
regions.sort(key=lambda r: -((r[0][2] - r[0][0]) * (r[0][3] - r[0][1])))  # draw large boxes first
for rep, nmonths, nbbox in regions:
    w, s, e, n = rep
    if nmonths >= full:
        color, ls, lw = "#2e7d32", "-", 1.6            # green: complete in time
    elif nmonths >= full * 0.5:
        color, ls, lw = "#ef8c00", (0, (5, 2)), 2.4    # orange: partial
    else:
        color, ls, lw = "#d32f2f", "--", 2.4           # red: severely incomplete
    ax.add_patch(mpatches.Rectangle((w, s), e - w, n - s, transform=ccrs.PlateCarree(),
                 facecolor=color, edgecolor=color, alpha=0.20, linewidth=lw, linestyle=ls, zorder=5))
    ax.add_patch(mpatches.Rectangle((w, s), e - w, n - s, transform=ccrs.PlateCarree(),
                 facecolor="none", edgecolor=color, linewidth=lw, linestyle=ls, zorder=6))
    tag = f"{nmonths}/{full}" + ("" if nbbox == 1 else f"\n({nbbox} extents)")
    ax.text((w + e) / 2, (s + n) / 2, tag, transform=ccrs.PlateCarree(), fontsize=7.5,
            ha="center", va="center", weight="bold", color="black", zorder=7,
            bbox=dict(boxstyle="round,pad=0.12", fc="white", ec="none", alpha=0.7))

legend = [
    Line2D([0], [0], color="#2e7d32", lw=3, label="Complete (132/132 months)"),
    Line2D([0], [0], color="#ef8c00", lw=3, ls=(0, (5, 2)), label="Partial (extent shifted; union still 132)"),
    Line2D([0], [0], color="#d32f2f", lw=3, ls="--", label="Severely incomplete (<50%)"),
]
ax.legend(handles=legend, loc="lower left", fontsize=10, framealpha=0.9)
ax.set_title(f"GMTDS downloaded coverage ({MODE}) — {len(regions)} sea regions (overlap-merged)\n"
             f"labels = unique months present in {min(YEARS)}-{max(YEARS)}", fontsize=13, weight="bold")
plt.tight_layout()
plt.show()


In [ ]:
import os, re, glob
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.lines import Line2D
import cartopy.crs as ccrs, cartopy.feature as cfeature
import rasterio
from rasterio.warp import transform_bounds
from shapely.geometry import box as sbox
from shapely.ops import unary_union

# ----------------- Config -----------------
BASE = rf"{DATA_ROOT}/climate_AIS_data/GMTDS/6_region"   # organized 6-region folder
YEARS = range(2011, 2022); FULL = 132                   # 2011-2021 should have 132 months
# 6 maritime region boxes, format (south, north, west, east)
TARGET_BOXES = {
    "NW_Europe":   [(43, 63, -15, 0), (48, 63, 0, 10)],
    "NW_Atlantic": [(18, 50, -98, -60)],
    "SE_Asia":     [(-10, 20, 95, 127)],
    "E_Asia":      [(20, 42, 117, 145)],
    "Med_Black":   [(30, 43, -5.5, 0), (30, 47.5, 0, 42)],
    "Baltic":      [(53, 66, 10, 30)],
}
# ----------------------------------------

MON = re.compile(r"_(\d{4})_(\d{2})")
def stem(name):  # strip year-month to get the 'source sea' name, for grouping
    return re.sub(r"_\d{4}_\d{2}.*", "", name)
def llbounds(ds):
    b = ds.bounds
    try: return transform_bounds(ds.crs, "EPSG:4326", b.left, b.bottom, b.right, b.top)
    except Exception: return (b.left, b.bottom, b.right, b.top)

# ---- scan each region: month set (completeness) + footprint (overlap) ----
info = {}
print(f"{'region':13s} {'months':>7}  status")
for region in TARGET_BOXES:
    tifs = glob.glob(os.path.join(BASE, region, "*.tif"))
    months = set()
    groups = {}
    for t in tifs:
        m = MON.search(os.path.basename(t))
        if not m: continue
        y, mo = int(m.group(1)), int(m.group(2))
        if y in YEARS: months.add((y, mo))
        groups.setdefault(stem(os.path.basename(t)), []).append((y, mo, t))
    # read footprint from first/last month of each source sea (catches extent drift)
    foot = []
    for g in groups.values():
        g.sort()
        for _, _, t in {g[0], g[-1]}:
            with rasterio.open(t) as ds:
                foot.append(sbox(*llbounds(ds)))
    n = len(months)
    miss = sorted(set((y, mo) for y in YEARS for mo in range(1, 13)) - months)
    info[region] = {"n": n, "foot": unary_union(foot) if foot else None, "missing": miss}
    status = "✅ 132/132 complete" if n == FULL else f"⚠ {n}/132  missing {len(miss)} months"
    print(f"{region:13s} {n:>7}  {status}")
    if miss: print(f"              missing: {miss[:12]}{' ...' if len(miss)>12 else ''}")

# ---- world map: boxes + data footprint overlap ----
fig = plt.figure(figsize=(20, 10.5))
ax = plt.axes(projection=ccrs.PlateCarree()); ax.set_global()
ax.add_feature(cfeature.LAND, facecolor="#f0ece1"); ax.add_feature(cfeature.OCEAN, facecolor="#dceaf2")
ax.add_feature(cfeature.COASTLINE, linewidth=0.4, edgecolor="#999")
ax.gridlines(draw_labels=True, linewidth=0.3, color="gray", alpha=0.35)

def fill(geom, **kw):
    if geom is None or geom.is_empty: return
    gs = list(geom.geoms) if geom.geom_type.startswith("Multi") else [geom]
    for g in gs: ax.fill(*g.exterior.xy, transform=ccrs.PlateCarree(), **kw)

for region, boxes in TARGET_BOXES.items():
    poly = unary_union([sbox(w, s, e, n) for (s, n, w, e) in boxes])
    foot = info[region]["foot"]
    overlap = poly.intersection(foot) if foot else None   # box area covered by data
    gap = poly.difference(foot) if foot else poly          # box area without data
    complete = info[region]["n"] == FULL
    fill(overlap, facecolor="#2e7d32", alpha=0.45, zorder=5)   # green = data overlap
    fill(gap,     facecolor="#d32f2f", alpha=0.40, zorder=5)   # red = no data in box
    for (s, n, w, e) in boxes:                                  # box edges
        ax.plot([w, e, e, w, w], [s, s, n, n, s], transform=ccrs.PlateCarree(),
                color="black", lw=1.8, zorder=7)
    b = poly.bounds
    ax.text((b[0]+b[2])/2, (b[1]+b[3])/2, f"{region}\n{info[region]['n']}/132",
            transform=ccrs.PlateCarree(), fontsize=10, ha="center", va="center", weight="bold",
            color=("#1b5e20" if complete else "#b71c1c"), zorder=8,
            bbox=dict(boxstyle="round,pad=0.2", fc="white", ec="black", alpha=0.85))

leg = [Line2D([0],[0], color="#2e7d32", lw=8, alpha=.5, label="downloaded data ∩ region (overlap)"),
       Line2D([0],[0], color="#d32f2f", lw=8, alpha=.5, label="no data inside region (gap)"),
       Line2D([0],[0], color="black", lw=1.8, label="6 maritime region boxes")]
plt.rcParams['font.sans-serif'] = ['Arial', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False
ax.legend(handles=leg, loc="lower left", fontsize=11, framealpha=.9)
ax.set_title("Downloaded GMTDS data vs 6 maritime regions overlap  (label = unique months 2011-2021 / 132)",
             fontsize=14, weight="bold")
plt.tight_layout(); plt.show()


In [ ]:
import pandas as pd

ACC = rf"{DATA_ROOT}/AIS_processing/accident_data/accident_cargo_2002_2021.parquet"
YEAR_RANGE = (2011, 2021)      # aligned with GMTDS; use (2002, 2021) to count all
EXCLUDE_TANKERS = False        # True = keep dry cargo only, excluding tankers/gas carriers

# Format (south, north, west, east)
TARGET_BOXES = {
    "NW_Europe":   [(43, 63, -15, 0), (48, 63, 0, 10)],
    "NW_Atlantic": [(18, 50, -98, -60)],
    "SE_Asia":     [(-10, 20, 95, 127)],
    "E_Asia":      [(20, 42, 117, 145)],
    "Med_Black":   [(30, 43, -5.5, 0), (30, 47.5, 0, 42)],
    "Baltic":      [(53, 66, 10, 30)],
}

df = pd.read_parquet(ACC)
d = df.dropna(subset=["lat", "lon"]).copy()
d = d[d["year"].between(*YEAR_RANGE)]
if EXCLUDE_TANKERS:
    mask = d["vessel_type"].str.contains("tanker|gas carrier|oil", case=False, na=False)
    d = d[~mask]

def region_of(lat, lon):
    for r, boxes in TARGET_BOXES.items():
        for s, n, w, e in boxes:
            if s <= lat <= n and w <= lon <= e:
                return r
    return "(outside)"

d["region"] = [region_of(a, o) for a, o in zip(d["lat"], d["lon"])]

order = list(TARGET_BOXES) + ["(outside)"]
counts = d["region"].value_counts().reindex(order, fill_value=0)
print(f"years {YEAR_RANGE}, exclude tankers={EXCLUDE_TANKERS}, records with coordinates {len(d)}\n")
print(counts.to_string())
print(f"\n6-region total: {counts[list(TARGET_BOXES)].sum()}")

# Optional: breakdown by ship type per region
print("\nTop ship types per region:")
print(d[d.region!="(outside)"].groupby("region")["vessel_type"].value_counts().groupby(level=0).head(3))
